# 🎬 "SHE" — 4K Hybrid Teaser First, Then 4K Full Film Master (Colab)
1. **Phase A (First):** Renders the **50s Hybrid Trailer** (Flow aerials + OG film, watermark-free, unified warm-gold typography) and upscales to **4K (`3840×2160`)** on Google Drive.
2. **Phase B (Second):** Automatically proceeds to render all **10 chunks (`10,680` frames)** of the full **7:25 SHE film** with Drive-backed resume, stitches the master, and exports **4K (`3840×2160`)**.

In [ ]:
# Step 1: Install dependencies & mount Google Drive
!apt-get update -qq && apt-get install -y -qq ffmpeg
!pip install -q opencv-python pillow numpy librosa

import os, sys, shutil, subprocess, importlib, multiprocessing as mp
from google.colab import drive
drive.mount('/content/drive')

DRIVE_4K_DIR = '/content/drive/MyDrive/SHE_Film_4K_Master'
DRIVE_CHUNKS_DIR = os.path.join(DRIVE_4K_DIR, 'chunks_1080p')
os.makedirs(DRIVE_CHUNKS_DIR, exist_ok=True)
print(f'✓ CPU cores: {mp.cpu_count()} | Drive output: {DRIVE_4K_DIR}')

In [ ]:
# Step 2: Clone repository (with all latest fixes, Flow botanical sprites & aerials)
REPO_BRANCH = 'codex/grounded-mushroom-colab'
subprocess.run(['rm', '-rf', '/content/aianimationautomate'], check=False)
subprocess.run(['git', '-c', 'http.version=HTTP/1.1', '-c', 'http.postBuffer=524288000', 'clone', '--depth', '1', '--branch', REPO_BRANCH, '--single-branch', 'https://github.com/sayan-bhattacharya/aianimationautomate.git', '/content/aianimationautomate'], check=True)

os.chdir('/content/aianimationautomate')
sys.path.insert(0, '/content/aianimationautomate')
importlib.invalidate_caches()
print('✓ Cloned branch:', REPO_BRANCH)

In [ ]:
# Step 2B (4K Asset Prep): Export 4K-upscaled AI image plates, dino sprites, and Flow drone clips to Drive
import glob, cv2
OUT_4K_ASSETS = os.path.join(DRIVE_4K_DIR, '4k_upscaled_assets')
os.makedirs(OUT_4K_ASSETS, exist_ok=True)
for pat in ['she_film/cache/buddha_*.jpg', 'she_film/cache/lab_*.jpg', 'she_film/cache/*dino*.png', 'she_film/cache/pokemon_*.png']:
    for p in sorted(glob.glob(pat)):
        img = cv2.imread(p, cv2.IMREAD_UNCHANGED)
        if img is None: continue
        h, w = img.shape[:2]
        scale = max(2, int(round(3840 / w))) if w > 400 else 4
        up = cv2.resize(img, (w * scale, h * scale), interpolation=cv2.INTER_LANCZOS4)
        cv2.imwrite(os.path.join(OUT_4K_ASSETS, os.path.basename(p)), up)
for mp4 in sorted(glob.glob('assets/flow_aerials/*.mp4')):
    out_mp4 = os.path.join(OUT_4K_ASSETS, os.path.basename(mp4))
    subprocess.run(['ffmpeg', '-y', '-i', mp4, '-vf', 'scale=3840:2160:flags=lanczos,unsharp=5:5:0.45:5:5:0.0', '-c:v', 'libx264', '-crf', '15', '-pix_fmt', 'yuv420p', '-c:a', 'copy', out_mp4], check=True)
print('✓ 4K AI image, dino, and Flow drone assets saved to:', OUT_4K_ASSETS)

In [ ]:
# Step 3 (FIRST): Render 50s Hybrid Teaser -> Upscale to 4K (3840x2160) & Save to Drive
TEASER_1080 = '/content/aianimationautomate/output_teaser/SHE_Teaser_50s_Hybrid_1080p.mp4'
TEASER_4K = os.path.join(DRIVE_4K_DIR, 'SHE_Teaser_50s_Hybrid_4K.mp4')

subprocess.run([sys.executable, 'render_hybrid_50s_trailer.py'], check=True)
subprocess.run([
    'ffmpeg', '-y', '-i', TEASER_1080,
    '-vf', 'scale=3840:2160:flags=lanczos,unsharp=5:5:0.45:5:5:0.0',
    '-c:v', 'libx264', '-preset', 'medium', '-crf', '16', '-pix_fmt', 'yuv420p',
    '-c:a', 'copy', TEASER_4K
], check=True)
print('✓ 4K Hybrid Teaser saved to Drive:', TEASER_4K)

In [ ]:
# Step 4 (SECOND): Render Full 7:25 SHE Film (10 Chunks) -> Stitch & Export 4K Master
import cv2, numpy as np
from she_film import frame, plates
from she_film.clock import DURATION, FPS, HEIGHT, TOTAL_FRAMES, WIDTH

subprocess.run(['python', '-m', 'she_film.generate_screens_cache'], check=True)
p = plates.load()
CHUNK_SIZE = 1068
n_chunks = (TOTAL_FRAMES + CHUNK_SIZE - 1) // CHUNK_SIZE
chunk_paths = []

for ci in range(n_chunks):
    f0 = ci * CHUNK_SIZE
    f1 = min(TOTAL_FRAMES, f0 + CHUNK_SIZE)
    c_path = os.path.join(DRIVE_CHUNKS_DIR, f'chunk_{ci+1:02d}.mp4')
    chunk_paths.append(c_path)
    if os.path.exists(c_path) and os.path.getsize(c_path) > 100000:
        print(f'Skipping completed chunk {ci+1:02d}')
        continue
    tmp_c = f'/content/chunk_{ci+1:02d}.mp4'
    vw = cv2.VideoWriter(tmp_c, cv2.VideoWriter_fourcc(*'mp4v'), FPS, (WIDTH, HEIGHT))
    for fi in range(f0, f1):
        rgb = frame.render(p, fi / FPS)
        vw.write(cv2.cvtColor(np.clip(rgb, 0, 255).astype(np.uint8), cv2.COLOR_RGB2BGR))
    vw.release()
    shutil.copy2(tmp_c, c_path)
    os.remove(tmp_c)
    print(f'✓ Saved chunk {ci+1:02d}/{n_chunks} to Drive')

concat_txt = '/content/concat_4k.txt'
with open(concat_txt, 'w') as f:
    for cp in chunk_paths:
        f.write(f"file '{cp}'\n")

MASTER_4K = os.path.join(DRIVE_4K_DIR, 'she_full_song_4K_crystal_nocturne_master.mp4')
subprocess.run([
    'ffmpeg', '-y', '-f', 'concat', '-safe', '0', '-i', concat_txt,
    '-i', '/content/aianimationautomate/She.wav',
    '-vf', 'scale=3840:2160:flags=lanczos,unsharp=5:5:0.45:5:5:0.0',
    '-c:v', 'libx264', '-preset', 'medium', '-crf', '16', '-pix_fmt', 'yuv420p',
    '-c:a', 'aac', '-b:a', '256k', '-shortest', MASTER_4K
], check=True)
print('✓ 4K Full Film Master saved to Drive:', MASTER_4K)